In [ ]:
%matplotlib notebook
from rfsoc_rfdc.rfsoc_overlay import RFSoCOverlay
from rfsoc_rfdc.rfdc_task import RfdcTask 
from rfsoc_rfdc.mts_task import MtsTask
from rfsoc_rfdc.array_calib_task import ArrayCalibTask
from rfsoc_rfdc.rfdc_config import ZCU216_CONFIG
import numpy as np
import matplotlib.pyplot as plt
import time

In [ ]:
ol = RFSoCOverlay(path_to_bitstream="./rfsoc_rfdc/bitstream/rfsoc_rfdc_v47_4t1r_bf.bit")
NEW_CONFIG ={
    "RefClockForPLL": 300.0,
    "DACSampleRate": 2400.0,
    "DACInterpolationRate": 4,
    "DACNCO": 700,
    "ADCSampleRate": 2400.0,
    "ADCInterpolationRate": 4,
    "ADCNCO": -700
}
ZCU216_CONFIG.update(NEW_CONFIG)

In [ ]:
rfdc_t = RfdcTask(ol, debug_mode=True, board="ZCU216")
mts_t = MtsTask(ol, board="ZCU216", debug_mode=True)

for task in [mts_t, rfdc_t]:
    task.start()
    task.join()

In [ ]:
num_channels=4

In [ ]:
calib_task = ArrayCalibTask(ol, num_tx_ch=1, num_dacs=num_channels, num_rx_ch=1, num_adcs=1)
calib_task.start()
calib_task.join()

In [ ]:
angle_offset = 0
start_angle, end_angle = (-90 + angle_offset), (90 + angle_offset)

In [ ]:
# Antenna pattern measurement
calib_task = ArrayCalibTask(ol, num_dacs=num_channels, num_adcs= 1)
# Start the TX tone
calib_task.tx_tone.start()
time.sleep(3)
# Sweep Calibrated
angles, powers = calib_task.sweep_pattern(start_angle=start_angle, end_angle=end_angle, step=1, calibrated=True)
# Sweep Uncalibrated
angles_uc, powers_uc = calib_task.sweep_pattern(start_angle=start_angle, end_angle=end_angle, step=1, calibrated=False)
# Stop the TX tone
calib_task.tx_tone.stop()

In [ ]:
from rfsoc_rfdc.plotter.radiation_plotter import RadiationPlotter, RadiationPatternData

plot_data = RadiationPatternData(angles=angles, powers=powers, powers_uncalib=powers_uc)

plotter = RadiationPlotter(num_channels=num_channels)
angles_np, powers_np = plotter.plot(plot_data)

# To plot from a saved file without running the experiment:
# angles_np, powers_np = plotter.plot_from_file('./exp_data/antenna_pattern_4T/antenna_pattern_4T.txt')

In [ ]:
plt.figure(figsize=(10,6))
plt.plot(angles_np, powers_np, marker='o')
plt.title('Antenna Radiation Pattern {num_channels}T1R')
plt.xlabel('Steering Angle (degrees)')
plt.ylabel('Measured Power (dBm/MHz)')
plt.grid(True)
plt.show()